In [1]:
FFMPEG_PATH = r"C:/ffmpeg/bin/ffmpeg.exe"
VIDEO_PATH = r".\data\2025-11-20_15-30-11-a3a383b4\95cbe6dd_0.0-323.503.mp4"
OUTPUT_DIR = r".\videos\velo_1"

FPS = 24
SCALE = 0.5
SIZE = (int(1600 * SCALE), int(1200 * SCALE))
START_FRAME = 1701
END_FRAME = 2100
NB_FRAMES = END_FRAME - START_FRAME

START_NUMBER = 0
QUALITY = 2

print(SIZE)


(800, 600)


# Images

In [2]:
import subprocess
from pathlib import Path

# Création du répertoire de sortie s'il n'existe pas
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

# Construction de la commande
cmd = [
    FFMPEG_PATH,
    "-i", VIDEO_PATH,
]

# Construction du filtre vidéo
filters = []

if FPS is not None:
    filters.append(f"fps={FPS}")

if END_FRAME is not None:
    filters.append(f"select='between(n,{START_FRAME},{END_FRAME})'")
else:
    filters.append(f"select='gte(n,{START_FRAME})'")

# Redimensionnement à 1280x720
filters.append(f"scale={SIZE[0]}:{SIZE[1]}")

# Important : réindexer les timestamps
filters.append("setpts=N/FRAME_RATE/TB")

cmd += ["-vf", ",".join(filters)]

cmd += [
    "-q:v", str(QUALITY),
    "-start_number", str(START_NUMBER),
    str(Path(OUTPUT_DIR) / "%05d.jpg")
]

print("Commande FFmpeg :")
print(" ".join(cmd))

process = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)

if process.returncode != 0:
    print("Erreur FFmpeg")
    print(process.stderr)
else:
    print("Extraction terminée avec succès")

Commande FFmpeg :
C:/ffmpeg/bin/ffmpeg.exe -i .\data\2025-11-20_15-30-11-a3a383b4\95cbe6dd_0.0-323.503.mp4 -vf fps=24,select='between(n,1701,2100)',scale=800:600,setpts=N/FRAME_RATE/TB -q:v 2 -start_number 0 videos\velo_1\%05d.jpg
Extraction terminée avec succès


# Charger les fixations

In [3]:
import os
# if using Apple MPS, fall back to CPU for unsupported ops
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
import numpy as np
import torch
import matplotlib.pyplot as plt
from PIL import Image
import pandas as pd

In [4]:
CSV_PATH = "./data/2025-11-20_15-30-11-a3a383b4/fixations.csv"
VIDEO_FPS = FPS


df = pd.read_csv(CSV_PATH)

t0 = df["start timestamp [ns]"].min()

df["time_s"] = (df["start timestamp [ns]"] - t0) / 1e9

df["frame"] = (df["time_s"] * VIDEO_FPS).astype(int)

fixations_extracted = df[[
    "fixation id",
    "frame",
    "fixation x [px]",
    "fixation y [px]",
    "duration [ms]"
]].rename(columns={
    "fixation x [px]": "x",
    "fixation y [px]": "y"
})

fixations_extracted.to_csv("fixations_with_frames.csv", index=False)


In [5]:
import pandas as pd

def get_fixation_triplets(df, min_frame):
    # Filtrer les frames >= 94 (inclus)
    filtered = df[df["frame"] >= min_frame].copy()

    # Décalage des frames
    filtered["frame"] = filtered["frame"] - min_frame
    filtered["x"] = (filtered["x"] * SCALE).astype(int)
    filtered["y"] = (filtered["y"] * SCALE).astype(int)

    triplets = list(
        zip(
            filtered["frame"].astype(int),
            filtered["x"].astype(float),
            filtered["y"].astype(float)
        )
    )

    return triplets


df = pd.read_csv("fixations_with_frames.csv")

triplets = get_fixation_triplets(df, min_frame=START_FRAME)


In [6]:
CSV_PATH = "./data/2025-11-20_15-30-11-a3a383b4/gaze.csv"

df = pd.read_csv(CSV_PATH)

# Temps de référence
t0 = df["timestamp [ns]"].min()

# Temps en secondes
df["time_s"] = (df["timestamp [ns]"] - t0) / 1e9

# Numéro de frame
df["frame"] = (df["time_s"] * VIDEO_FPS).astype(int)

# Positions du regard pour toutes les frames
gaze_per_sample = df[[
    "frame",
    "gaze x [px]",
    "gaze y [px]"
]].rename(columns={
    "gaze x [px]": "x",
    "gaze y [px]": "y"
})

gaze_per_sample.to_csv("gaze_with_frames.csv", index=False)


In [7]:
import pandas as pd

def get_gaze_triplets(df, min_frame):
    # Filtrer les frames >= min_frame
    filtered = df[df["frame"] >= min_frame].copy()

    # Ne garder que la première occurrence de chaque frame
    filtered = filtered.drop_duplicates(subset="frame", keep="first")

    # Décalage des frames
    filtered["frame"] = filtered["frame"] - min_frame

    # Mise à l'échelle
    filtered["x"] = (filtered["x"] * SCALE).astype(int)
    filtered["y"] = (filtered["y"] * SCALE).astype(int)

    triplets = list(
        zip(
            filtered["frame"].astype(int),
            filtered["x"].astype(float),
            filtered["y"].astype(float)
        )
    )

    return triplets


df = pd.read_csv("gaze_with_frames.csv")

triplets_gaze = get_gaze_triplets(df, min_frame=START_FRAME)


# Run sam2

In [8]:
# select the device for computation
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"using device: {device}")

if device.type == "cuda":
    # use bfloat16 for the entire notebook
    torch.autocast("cuda", dtype=torch.bfloat16).__enter__()
    # turn on tfloat32 for Ampere GPUs (https://pytorch.org/docs/stable/notes/cuda.html#tensorfloat-32-tf32-on-ampere-devices)
    if torch.cuda.get_device_properties(0).major >= 8:
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
elif device.type == "mps":
    print(
        "\nSupport for MPS devices is preliminary. SAM 2 is trained with CUDA and might "
        "give numerically different outputs and sometimes degraded performance on MPS. "
        "See e.g. https://github.com/pytorch/pytorch/issues/84936 for a discussion."
    )

using device: cuda


In [9]:
from sam2.build_sam import build_sam2_video_predictor

sam2_checkpoint = os.path.abspath("./model/sam2.1_hiera_small.pt")
model_cfg = os.path.abspath("./model/sam2.1_hiera_s.yaml")

predictor = build_sam2_video_predictor(model_cfg, sam2_checkpoint, device=device)

In [10]:
# `video_dir` a directory of JPEG frames with filenames like `<frame_index>.jpg`
video_dir = "./videos/velo_1"

# scan all the JPEG frame names in this directory
frame_names = [
    p for p in os.listdir(video_dir)
    if os.path.splitext(p)[-1] in [".jpg", ".jpeg", ".JPG", ".JPEG"]
]
frame_names.sort(key=lambda p: int(os.path.splitext(p)[0]))

#### Initialize the inference state

In [11]:
inference_state = predictor.init_state(video_path=video_dir)

frame loading (JPEG): 100%|██████████| 401/401 [00:07<00:00, 51.30it/s]


In [12]:
predictor.reset_state(inference_state)

In [13]:
obj_id = 1 

for ann_frame_idx, x, y in triplets:
    if ann_frame_idx > NB_FRAMES:
        break

    print(f"Adding gaze point at frame {ann_frame_idx}, x={x}, y={y}")

    points = np.array([[x, y]], dtype=np.float32)
    labels = np.array([1], np.int32)

    _, out_obj_ids, out_mask_logits = predictor.add_new_points_or_box(
        inference_state=inference_state,
        frame_idx=ann_frame_idx,
        obj_id=obj_id,
        points=points,
        labels=labels,
    )


Adding gaze point at frame 11, x=412.0, y=205.0
Adding gaze point at frame 16, x=422.0, y=229.0
Adding gaze point at frame 20, x=439.0, y=125.0
Adding gaze point at frame 25, x=449.0, y=128.0
Adding gaze point at frame 31, x=285.0, y=149.0
Adding gaze point at frame 36, x=282.0, y=159.0


C:\Users\romai\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sam2\sam2_video_predictor.py:786: UserWarning: cannot import name '_C' from 'sam2' (C:\Users\romai\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sam2\__init__.py)

Skipping the post-processing step due to the error above. You can still use SAM 2 and it's OK to ignore the error above, although some post-processing functionality may be limited (which doesn't affect the results in most cases; see https://github.com/facebookresearch/sam2/blob/main/INSTALL.md).
  pred_masks_gpu = fill_holes_in_mask_scores(


Adding gaze point at frame 41, x=469.0, y=135.0
Adding gaze point at frame 46, x=448.0, y=141.0
Adding gaze point at frame 53, x=385.0, y=148.0
Adding gaze point at frame 59, x=400.0, y=149.0
Adding gaze point at frame 65, x=427.0, y=134.0
Adding gaze point at frame 72, x=386.0, y=144.0
Adding gaze point at frame 78, x=434.0, y=130.0
Adding gaze point at frame 82, x=432.0, y=117.0
Adding gaze point at frame 87, x=440.0, y=106.0
Adding gaze point at frame 90, x=433.0, y=121.0
Adding gaze point at frame 95, x=434.0, y=128.0
Adding gaze point at frame 101, x=413.0, y=155.0
Adding gaze point at frame 110, x=342.0, y=119.0
Adding gaze point at frame 119, x=323.0, y=153.0
Adding gaze point at frame 126, x=461.0, y=148.0
Adding gaze point at frame 134, x=363.0, y=145.0
Adding gaze point at frame 147, x=380.0, y=115.0
Adding gaze point at frame 156, x=420.0, y=135.0
Adding gaze point at frame 159, x=428.0, y=122.0
Adding gaze point at frame 166, x=402.0, y=149.0
Adding gaze point at frame 173,

In [14]:
video_segments = {}

with torch.no_grad():
    for out_frame_idx, out_obj_ids, out_mask_logits in predictor.propagate_in_video(inference_state):

        # Seuil sur GPU (rapide)
        masks = out_mask_logits > 0.0  # bool tensor [N, H, W]

        # ⚠️ UNE SEULE sync GPU → CPU
        masks_cpu = masks.cpu().numpy()

        frame_dict = {}
        for i, out_obj_id in enumerate(out_obj_ids):
            frame_dict[out_obj_id] = masks_cpu[i]

        video_segments[out_frame_idx] = frame_dict

propagate in video: 100%|██████████| 390/390 [01:40<00:00,  3.88it/s]


In [15]:
import cv2
import numpy as np
from PIL import Image
import os

# Fonction pour superposer le masque sur l'image
def overlay_mask(image, mask, color=(0, 0, 255), alpha=0.5):
    # S'assurer que le masque a la bonne forme
    if mask.ndim == 3:
        mask = mask[0]  # parfois c'est (1, H, W)
    if mask.shape != image.shape[:2]:
        # redimensionner le masque si nécessaire
        mask = cv2.resize(mask.astype(np.uint8), (image.shape[1], image.shape[0]))
        mask = mask.astype(bool)
    
    overlay = image.copy()
    overlay[mask] = (np.array(color) * alpha + overlay[mask] * (1 - alpha)).astype(np.uint8)
    return overlay

# Récupérer les dimensions exactes de la première image
img0 = np.array(Image.open(os.path.join(video_dir, frame_names[0])).convert("RGB"))
height, width = img0.shape[:2]

# Paramètres vidéo
video_out_path = f"./export/video_with_masks_{START_FRAME}_{END_FRAME}.mp4"
fps = FPS
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
video_writer = cv2.VideoWriter(video_out_path, fourcc, fps, (width, height))

# Boucle sur toutes les frames
for idx, frame_name in enumerate(frame_names):
    img = np.array(Image.open(os.path.join(video_dir, frame_names[idx])).convert("RGB"))
    img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
    
    if idx in video_segments:
        for out_obj_id, out_mask in video_segments[idx].items():
            img = overlay_mask(img, out_mask)
    
    # Ajouter le point de regard si disponible
    _, x, y = triplets_gaze[idx]
    cv2.circle(img, (int(x), int(y)), radius=10, color=(0, 255, 0), thickness=-1)
    video_writer.write(img)

video_writer.release()
print(f"Vidéo enregistrée dans {video_out_path}")


Vidéo enregistrée dans ./export/video_with_masks_1701_2100.mp4
